In [1]:
import sys
sys.path.append('../')

In [2]:
import pandas as pd

In [3]:
from transform import get_state_map_long_to_short, transform_gen
from load_dims import get_engine, get_all_maps

In [4]:
from transform import transform_prices

In [5]:
"""
WARNING: gen: missing fuel_id
  WARNING: prices: missing sector_id
"""

'\nWARNING: gen: missing fuel_id\n  WARNING: prices: missing sector_id\n'

In [6]:
engine = get_engine()

Connection successful: (1,)


In [7]:
maps = get_all_maps(engine)

In [8]:
maps["fuel"]

{'ALL': 87,
 'AOR': 88,
 'BIO': 89,
 'COL': 90,
 'COW': 91,
 'DFO': 92,
 'FOS': 93,
 'GEO': 94,
 'HPS': 95,
 'HYC': 96,
 'LFG': 97,
 'LIG': 98,
 'MLG': 99,
 'NG': 100,
 'NGO': 101,
 'NUC': 102,
 'OOG': 103,
 'ORW': 104,
 'OTH': 105,
 'PEL': 106,
 'PET': 107,
 'REN': 108,
 'RFO': 109,
 'SPV': 110,
 'SUB': 111,
 'SUN': 112,
 'WAS': 113,
 'WND': 114,
 'WNT': 115,
 'WOC': 116,
 'WOO': 117,
 'WWW': 118,
 'MSB': 119,
 'PC': 120,
 'RC': 121,
 'STH': 122,
 'BIS': 123,
 'BIT': 124,
 'DPV': 125,
 'TPV': 126,
 'TSN': 127,
 'ANT': 128,
 'WNS': 129}

In [9]:
price_df = pd.read_csv("../data/price_data.csv")
price_df.head()

,period,stateid,stateDescription,sectorid,sectorName,price,price-units
0,2025-03,AK,Alaska,ALL,all sectors,22.96,cents per kilowatt-hour
1,2025-03,AK,Alaska,COM,commercial,22.10,cents per kilowatt-hour
2,2025-03,AK,Alaska,IND,industrial,20.17,cents per kilowatt-hour
3,2025-03,AK,Alaska,OTH,other,NaN,cents per kilowatt-hour
4,2025-03,AK,Alaska,RES,residential,25.79,cents per kilowatt-hour


In [10]:
gen_df = pd.read_csv("../data/gen_data.csv")
gen_df.head()

,period,location,stateDescription,sectorid,sectorDescription,fueltypeid,fuelTypeDescription,generation,generation-units
0,2025-03,90,Pacific,1,Electric Utility,ALL,all fuels,17842.73950,thousand megawatthours
1,2025-03,90,Pacific,1,Electric Utility,AOR,all renewables,822.00528,thousand megawatthours
2,2025-03,90,Pacific,1,Electric Utility,BIO,biomass,37.41079,thousand megawatthours
3,2025-03,90,Pacific,1,Electric Utility,COL,"coal, excluding waste coal",36.42772,thousand megawatthours
4,2025-03,90,Pacific,1,Electric Utility,COW,all coal products,36.42772,thousand megawatthours


In [11]:
long_to_short = get_state_map_long_to_short(price_df)

In [12]:
gen_clean = transform_gen(gen_df, long_to_short, maps)

In [13]:
gen_clean.head()

,date_id,state_id,fuel_id,generation
416,202503,125,87,410.46621
417,202503,125,88,9.51077
418,202503,125,90,36.42772
419,202503,125,91,36.42772
420,202503,125,92,47.27726


In [14]:
gen_df["fuel_id"] = gen_df.fueltypeid.map(maps["fuel"])
gen_df.head()

,period,location,stateDescription,sectorid,sectorDescription,fueltypeid,fuelTypeDescription,generation,generation-units,fuel_id
0,2025-03,90,Pacific,1,Electric Utility,ALL,all fuels,17842.73950,thousand megawatthours,87.0
1,2025-03,90,Pacific,1,Electric Utility,AOR,all renewables,822.00528,thousand megawatthours,88.0
2,2025-03,90,Pacific,1,Electric Utility,BIO,biomass,37.41079,thousand megawatthours,89.0
3,2025-03,90,Pacific,1,Electric Utility,COL,"coal, excluding waste coal",36.42772,thousand megawatthours,90.0
4,2025-03,90,Pacific,1,Electric Utility,COW,all coal products,36.42772,thousand megawatthours,91.0


In [15]:
gen_nulls = gen_df.loc[gen_df["fuel_id"].isnull()]
len(gen_nulls)

2298

In [16]:
gen_nulls.fuelTypeDescription.unique()

<StringArray>
['biomass']
Length: 1, dtype: str

In [17]:
gen_nulls.head()

,period,location,stateDescription,sectorid,sectorDescription,fueltypeid,fuelTypeDescription,generation,generation-units,fuel_id
16,2025-03,90,Pacific,1,Electric Utility,OB2,biomass,3.70237,thousand megawatthours,NaN
17,2025-03,90,Pacific,1,Electric Utility,OBW,biomass,3.70237,thousand megawatthours,NaN
48,2025-03,90,Pacific,2,IPP Non-CHP,OB2,biomass,30.73068,thousand megawatthours,NaN
49,2025-03,90,Pacific,2,IPP Non-CHP,OBW,biomass,36.37543,thousand megawatthours,NaN
80,2025-03,90,Pacific,3,IPP CHP,OB2,biomass,NaN,thousand megawatthours,NaN


In [18]:
price_df.head()

,period,stateid,stateDescription,sectorid,sectorName,price,price-units
0,2025-03,AK,Alaska,ALL,all sectors,22.96,cents per kilowatt-hour
1,2025-03,AK,Alaska,COM,commercial,22.10,cents per kilowatt-hour
2,2025-03,AK,Alaska,IND,industrial,20.17,cents per kilowatt-hour
3,2025-03,AK,Alaska,OTH,other,NaN,cents per kilowatt-hour
4,2025-03,AK,Alaska,RES,residential,25.79,cents per kilowatt-hour


In [19]:
price_df["sector_id"] = price_df.sectorName.map(maps["sector"])
price_df.head()

,period,stateid,stateDescription,sectorid,sectorName,price,price-units,sector_id
0,2025-03,AK,Alaska,ALL,all sectors,22.96,cents per kilowatt-hour,13
1,2025-03,AK,Alaska,COM,commercial,22.10,cents per kilowatt-hour,14
2,2025-03,AK,Alaska,IND,industrial,20.17,cents per kilowatt-hour,15
3,2025-03,AK,Alaska,OTH,other,NaN,cents per kilowatt-hour,16
4,2025-03,AK,Alaska,RES,residential,25.79,cents per kilowatt-hour,17


In [20]:
price_nulls = price_df.loc[price_df["sector_id"].isnull()]
len(price_nulls)

0

In [21]:
prices_clean = transform_prices(price_df, maps)
prices_clean.head()

,date_id,state_id,sector_id,price_per_kwh
0,202503,125,13,22.96
1,202503,125,14,22.10
2,202503,125,15,20.17
3,202503,125,16,NaN
4,202503,125,17,25.79


In [22]:
prices_clean.sector_id.unique()

array([13, 14, 15, 16, 17, 18])

In [23]:
maps["sector"]

{'all sectors': 13,
 'commercial': 14,
 'industrial': 15,
 'other': 16,
 'residential': 17,
 'transportation': 18}

### Checking dupes and unique constraints

In [24]:
gen_clean.head()

,date_id,state_id,fuel_id,generation
416,202503,125,87,410.46621
417,202503,125,88,9.51077
418,202503,125,90,36.42772
419,202503,125,91,36.42772
420,202503,125,92,47.27726


In [28]:
gen_dupes = gen_clean[gen_clean.duplicated(["date_id", "state_id", "fuel_id", "generation"])]
len(gen_dupes)

18955

In [31]:
gen_dupes.head(20)

,date_id,state_id,fuel_id,generation
444,202503,125,110,NaN
445,202503,125,112,NaN
479,202503,125,88,0.00000
489,202503,125,108,0.00000
492,202503,125,118,0.00000
493,202503,125,87,NaN
512,202503,125,117,0.00000
518,202503,125,90,36.42772
519,202503,125,91,36.42772
520,202503,125,92,47.27726


In [30]:
gen_dupes.generation.unique()

array([       nan,    0.     ,   36.42772, ..., 3028.47603, 1372.29969,
       1244.26045], shape=(4323,))